In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código para calcular correlación entre estaciones para una variable dada (por defecto O3).
Lee archivos CSV de una carpeta (por ejemplo, los originales o los imputados),
alinea las series por fecha y calcula la matriz de correlación.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN (modifica según tu caso)
# ============================================================================

# Carpeta donde están los CSV de cada estación (puede ser la original o imputed_global)
DATA_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/imputed_global/")
# Si quieres usar los datos por transecto (un solo archivo por transecto), cambia a:
# DATA_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/")
# Y luego cada archivo contiene varias estaciones; el código manejará ambos casos.

# Variable a correlacionar (columna)
VAR = 'O3'

# Método de correlación: 'pearson', 'spearman', 'kendall'
CORR_METHOD = 'pearson'

# Umbral mínimo de datos comunes (fracción) para considerar un par de estaciones
MIN_COMMON_FRAC = 0.3   # al menos 50% de horas en común

# Guardar gráficos y tablas
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/correlaciones_estaciones")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath, var):
    """Carga un CSV, asume índice datetime, devuelve Serie con la variable."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if var not in df.columns:
        raise ValueError(f"Variable '{var}' no encontrada en {filepath}")
    serie = df[var].copy()
    # Forzar tipo numérico
    serie = pd.to_numeric(serie, errors='coerce')
    return serie

def get_series_from_folder(folder_path, var):
    """
    Recorre todos los archivos CSV en folder_path y devuelve un dict {nombre: Serie}.
    Si el archivo contiene varias estaciones (columna 'Estacion'), extrae cada una.
    """
    series_dict = {}
    files = list(Path(folder_path).glob("*.csv"))
    for f in files:
        name = f.stem
        df = pd.read_csv(f, index_col=0, parse_dates=True)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
        # Verificar si tiene columna 'Estacion' (caso por transecto)
        if 'Estacion' in df.columns:
            # Agrupar por estación
            for estacion, group in df.groupby('Estacion'):
                if var in group.columns:
                    serie = group[var].copy()
                    serie = pd.to_numeric(serie, errors='coerce')
                    # El índice de group puede tener duplicados? asumimos que no.
                    series_dict[f"{name}_{estacion}"] = serie
        else:
            # Archivo de una sola estación
            if var in df.columns:
                serie = df[var].copy()
                serie = pd.to_numeric(serie, errors='coerce')
                series_dict[name] = serie
    return series_dict

def align_series(series_dict):
    """Alinea todas las series por índice de tiempo (unión externa) y devuelve DataFrame."""
    df_all = pd.concat(series_dict, axis=1)
    # Renombrar columnas si son MultiIndex (por si acaso)
    df_all.columns = [col if isinstance(col, str) else col[1] for col in df_all.columns]
    return df_all

def compute_correlation_matrix(df, method='pearson', min_common_frac=0.5):
    """
    Calcula matriz de correlación entre columnas, ignorando pares con pocos datos comunes.
    """
    n = len(df)
    min_common = int(n * min_common_frac)
    corr_matrix = pd.DataFrame(index=df.columns, columns=df.columns, dtype=float)
    for i, col1 in enumerate(df.columns):
        for j, col2 in enumerate(df.columns):
            if i == j:
                corr_matrix.loc[col1, col2] = 1.0
                continue
            # Datos no nulos en ambos
            mask = df[col1].notna() & df[col2].notna()
            if mask.sum() < min_common:
                corr_matrix.loc[col1, col2] = np.nan
            else:
                corr_matrix.loc[col1, col2] = df[col1][mask].corr(df[col2][mask], method=method)
    return corr_matrix

def plot_correlation_heatmap(corr_matrix, title, save_path):
    """Dibuja mapa de calor de la matriz de correlación."""
    plt.figure(figsize=(12, 10))
    mask = corr_matrix.isna()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, mask=mask,
                cbar_kws={'shrink': 0.8})
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Gráfico guardado: {save_path}")

def plot_correlation_clustermap(corr_matrix, title, save_path):
    """Mapa de calor con clustering jerárquico."""
    if corr_matrix.isna().all().all():
        print("  Matriz vacía, no se puede generar clustermap.")
        return
    # Reemplazar NaN por 0 para que no rompa el clustering (pero puede distorsionar)
    corr_filled = corr_matrix.fillna(0)
    g = sns.clustermap(corr_filled, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       figsize=(12, 10), linewidths=0.5,
                       cbar_kws={'shrink': 0.8})
    g.ax_heatmap.set_title(title)
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Clustermap guardado: {save_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print(f"Cálculo de correlación ({CORR_METHOD}) para variable '{VAR}'")
    print("="*60)

    # 1. Cargar series
    print(f"\nCargando datos desde: {DATA_DIR}")
    series = get_series_from_folder(DATA_DIR, VAR)
    if not series:
        print("No se encontraron series válidas. Revise la carpeta.")
        exit()
    print(f"Estaciones encontradas: {list(series.keys())}")

    # 2. Alinear temporalmente
    df_aligned = align_series(series)
    print(f"Periodo: {df_aligned.index.min()} a {df_aligned.index.max()}")
    print(f"Total de horas: {len(df_aligned)}")

    # 3. Calcular matriz de correlación
    corr_matrix = compute_correlation_matrix(df_aligned, method=CORR_METHOD,
                                              min_common_frac=MIN_COMMON_FRAC)
    # Guardar matriz en CSV
    corr_csv = os.path.join(OUTPUT_DIR, f"correlacion_{VAR}_{CORR_METHOD}.csv")
    corr_matrix.to_csv(corr_csv)
    print(f"\nMatriz de correlación guardada en: {corr_csv}")

    # 4. Generar gráficos
    title = f"Correlación {CORR_METHOD} de {VAR} entre estaciones"
    plot_correlation_heatmap(corr_matrix, title,
                             os.path.join(OUTPUT_DIR, f"heatmap_{VAR}_{CORR_METHOD}.png"))
    plot_correlation_clustermap(corr_matrix, title,
                                os.path.join(OUTPUT_DIR, f"clustermap_{VAR}_{CORR_METHOD}.png"))

    # 5. Mostrar las 10 correlaciones más altas (fuera de la diagonal)
    # Obtener matriz triangular superior sin diagonal
    corr_triu = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    corr_triu = corr_triu.unstack().dropna().sort_values(ascending=False)
    if not corr_triu.empty:
        print("\nTop 10 pares con mayor correlación:")
        for (col1, col2), val in corr_triu.head(10).items():
            print(f"  {col1} vs {col2}: {val:.3f}")
    else:
        print("\nNo hay pares con suficientes datos comunes.")

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Cálculo de correlación (pearson) para variable 'O3'

Cargando datos desde: /Volumes/copia seguridad1/enviar_benja/imputed_global/
Estaciones encontradas: ['T1_E1_Alicante_Estacion 1', 'T1_E2_Elda_Estacion 2', 'T1_E2_Elda_T1_E2_Elda', 'T2_E1_Elche_Estacion 1', 'T2_E1_Elche_T2_E1_Elche', 'T2_E2_Elda_Estacion 2', 'T2_E2_Elda_T2_E2_Elda', 'T3_E1_Valencia_Estacion 1', 'T3_E2_Buñol_Estacion 2', 'T4_E1_Valencia_Estacion 1', 'T4_E2_Villar_Arzobispo_Estacion 2', 'T6_E1_Castellon_Estacion 1', 'T6_E1_Castellon_T6_E1_Castellon', 'T6_E2_Onda_Estacion 2', 'T7_E1_Sant_Jordi_Estacion 1', 'T7_E2_Coratxa_Estacion 2', 'T7_E3_Zorita_Estacion 3', 'T8_E1_Sant_Jordi_Estacion 1', 'T8_E2_Morella_Estacion 2', 'T8_E3_Zorita_Estacion 3']


ValueError: cannot reindex on an axis with duplicate labels

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código para calcular correlación entre estaciones para una variable dada (por defecto O3).
Lee archivos CSV de una carpeta (por ejemplo, los originales o los imputados),
alinea las series por fecha y calcula la matriz de correlación.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN (modifica según tu caso)
# ============================================================================

# Carpeta donde están los CSV de cada estación (puede ser la original o imputed_global)
DATA_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/imputed_global/")

# Variable a correlacionar (columna)
VAR = 'O3'

# Método de correlación: 'pearson', 'spearman', 'kendall'
CORR_METHOD = 'pearson'

# Umbral mínimo de datos comunes (fracción) para considerar un par de estaciones
MIN_COMMON_FRAC = 0.3   # al menos 30% de horas en común

# Guardar gráficos y tablas
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/correlaciones_estaciones")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def clean_series(serie):
    """
    Asegura que la serie tenga índice único y ordenado.
    - Elimina duplicados de índice (conserva el primero)
    - Ordena por índice
    - Convierte a numérico (coerce)
    """
    # Forzar tipo numérico
    serie = pd.to_numeric(serie, errors='coerce')
    # Eliminar duplicados de índice (keep='first' mantiene el primero)
    if not serie.index.is_unique:
        serie = serie[~serie.index.duplicated(keep='first')]
    # Ordenar índice
    serie = serie.sort_index()
    return serie

def load_station_data(filepath, var):
    """Carga un CSV, asume índice datetime, devuelve Serie limpia con la variable."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if var not in df.columns:
        raise ValueError(f"Variable '{var}' no encontrada en {filepath}")
    serie = df[var].copy()
    serie = clean_series(serie)
    return serie

def get_series_from_folder(folder_path, var):
    """
    Recorre todos los archivos CSV en folder_path y devuelve un dict {nombre: Serie}.
    Si el archivo contiene varias estaciones (columna 'Estacion'), extrae cada una.
    Cada serie es limpiada (índice único y ordenado).
    """
    series_dict = {}
    files = list(Path(folder_path).glob("*.csv"))
    for f in files:
        name = f.stem
        df = pd.read_csv(f, index_col=0, parse_dates=True)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
        # Verificar si tiene columna 'Estacion' (caso por transecto)
        if 'Estacion' in df.columns:
            # Agrupar por estación
            for estacion, group in df.groupby('Estacion'):
                if var in group.columns:
                    serie = group[var].copy()
                    serie = clean_series(serie)
                    series_dict[f"{name}_{estacion}"] = serie
        else:
            # Archivo de una sola estación
            if var in df.columns:
                serie = df[var].copy()
                serie = clean_series(serie)
                series_dict[name] = serie
    return series_dict

def align_series(series_dict):
    """Alinea todas las series por índice de tiempo (unión externa) y devuelve DataFrame."""
    # Asegurar que todas las series tienen índice ordenado y único (ya lo hicimos en get_series)
    # Concatenar
    df_all = pd.concat(series_dict, axis=1)
    # Renombrar columnas si son MultiIndex (por si acaso)
    df_all.columns = [col if isinstance(col, str) else col[1] for col in df_all.columns]
    return df_all

def compute_correlation_matrix(df, method='pearson', min_common_frac=0.5):
    """
    Calcula matriz de correlación entre columnas, ignorando pares con pocos datos comunes.
    """
    n = len(df)
    min_common = int(n * min_common_frac)
    corr_matrix = pd.DataFrame(index=df.columns, columns=df.columns, dtype=float)
    for i, col1 in enumerate(df.columns):
        for j, col2 in enumerate(df.columns):
            if i == j:
                corr_matrix.loc[col1, col2] = 1.0
                continue
            # Datos no nulos en ambos
            mask = df[col1].notna() & df[col2].notna()
            if mask.sum() < min_common:
                corr_matrix.loc[col1, col2] = np.nan
            else:
                corr_matrix.loc[col1, col2] = df[col1][mask].corr(df[col2][mask], method=method)
    return corr_matrix

def plot_correlation_heatmap(corr_matrix, title, save_path):
    """Dibuja mapa de calor de la matriz de correlación."""
    plt.figure(figsize=(12, 10))
    mask = corr_matrix.isna()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, mask=mask,
                cbar_kws={'shrink': 0.8})
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Gráfico guardado: {save_path}")

def plot_correlation_clustermap(corr_matrix, title, save_path):
    """Mapa de calor con clustering jerárquico."""
    if corr_matrix.isna().all().all():
        print("  Matriz vacía, no se puede generar clustermap.")
        return
    # Reemplazar NaN por 0 para que no rompa el clustering (pero puede distorsionar)
    corr_filled = corr_matrix.fillna(0)
    g = sns.clustermap(corr_filled, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       figsize=(12, 10), linewidths=0.5,
                       cbar_kws={'shrink': 0.8})
    g.ax_heatmap.set_title(title)
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Clustermap guardado: {save_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print(f"Cálculo de correlación ({CORR_METHOD}) para variable '{VAR}'")
    print("="*60)

    # 1. Cargar series
    print(f"\nCargando datos desde: {DATA_DIR}")
    series = get_series_from_folder(DATA_DIR, VAR)
    if not series:
        print("No se encontraron series válidas. Revise la carpeta.")
        exit()
    print(f"Estaciones encontradas: {list(series.keys())}")

    # 2. Alinear temporalmente
    df_aligned = align_series(series)
    print(f"Periodo: {df_aligned.index.min()} a {df_aligned.index.max()}")
    print(f"Total de horas: {len(df_aligned)}")

    # 3. Calcular matriz de correlación
    corr_matrix = compute_correlation_matrix(df_aligned, method=CORR_METHOD,
                                              min_common_frac=MIN_COMMON_FRAC)
    # Guardar matriz en CSV
    corr_csv = os.path.join(OUTPUT_DIR, f"correlacion_{VAR}_{CORR_METHOD}.csv")
    corr_matrix.to_csv(corr_csv)
    print(f"\nMatriz de correlación guardada en: {corr_csv}")

    # 4. Generar gráficos
    title = f"Correlación {CORR_METHOD} de {VAR} entre estaciones"
    plot_correlation_heatmap(corr_matrix, title,
                             os.path.join(OUTPUT_DIR, f"heatmap_{VAR}_{CORR_METHOD}.png"))
    plot_correlation_clustermap(corr_matrix, title,
                                os.path.join(OUTPUT_DIR, f"clustermap_{VAR}_{CORR_METHOD}.png"))

    # 5. Mostrar las 10 correlaciones más altas (fuera de la diagonal)
    # Obtener matriz triangular superior sin diagonal
    corr_triu = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    corr_triu = corr_triu.unstack().dropna().sort_values(ascending=False)
    if not corr_triu.empty:
        print("\nTop 10 pares con mayor correlación:")
        for (col1, col2), val in corr_triu.head(10).items():
            print(f"  {col1} vs {col2}: {val:.3f}")
    else:
        print("\nNo hay pares con suficientes datos comunes.")

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Cálculo de correlación (pearson) para variable 'O3'

Cargando datos desde: /Volumes/copia seguridad1/enviar_benja/imputed_global/
Estaciones encontradas: ['T1_E1_Alicante_Estacion 1', 'T1_E2_Elda_Estacion 2', 'T1_E2_Elda_T1_E2_Elda', 'T2_E1_Elche_Estacion 1', 'T2_E1_Elche_T2_E1_Elche', 'T2_E2_Elda_Estacion 2', 'T2_E2_Elda_T2_E2_Elda', 'T3_E1_Valencia_Estacion 1', 'T3_E2_Buñol_Estacion 2', 'T4_E1_Valencia_Estacion 1', 'T4_E2_Villar_Arzobispo_Estacion 2', 'T6_E1_Castellon_Estacion 1', 'T6_E1_Castellon_T6_E1_Castellon', 'T6_E2_Onda_Estacion 2', 'T7_E1_Sant_Jordi_Estacion 1', 'T7_E2_Coratxa_Estacion 2', 'T7_E3_Zorita_Estacion 3', 'T8_E1_Sant_Jordi_Estacion 1', 'T8_E2_Morella_Estacion 2', 'T8_E3_Zorita_Estacion 3']
Periodo: 2006-01-01 00:00:00 a 2068-01-01 00:00:00
Total de horas: 320214

Matriz de correlación guardada en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/correlaciones_estaciones/correlacion_O3_pearson.csv
  Gráfico guardado: /Users/benjamincarbonell/Documents/GitHub/TFGF